# MJ AI — Large Dataset LoRA Training (1 Million Examples)

**Important:** This notebook prepares a real training run; it does not fabricate one million examples. You must provide at least 1,000,000 real, high-quality, permissioned chat examples as JSONL in Google Drive. Standard free Colab GPU availability and runtime limits are not guaranteed; one million examples may require a paid or dedicated GPU environment and multiple sessions. Fine-tuning a 0.5B model does not make it equivalent to ChatGPT.

Expected format: one JSON object per line, each with `messages` containing `user` and `assistant` messages. Example: `{"messages":[{"role":"user","content":"Make a 15-second reel script"},{"role":"assistant","content":"0–2s hook..."}]}`.

In [ ]:
!pip -q install -U transformers datasets peft accelerate safetensors
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, torch
REPO='/content/Ai-video-free'
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','https://github.com/singwithsashu-a11y/Ai-video-free.git',REPO],check=True)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('A GPU runtime is strongly recommended. In Colab choose Runtime > Change runtime type > T4 GPU if available; access is not guaranteed.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Put your JSONL dataset in Google Drive
Create `/MyDrive/mj-ai-data/mj_ai_1m.jsonl`. Use data you own or have permission to use; remove secrets and private/sensitive chats. Do not put the one-million-row dataset in GitHub because it may exceed repository limits.

In [ ]:
DATA='/content/drive/MyDrive/mj-ai-data/mj_ai_1m.jsonl'
assert os.path.isfile(DATA), f'Dataset not found: {DATA}'
def count_lines(path):
    with open(path, 'rb') as f: return sum(1 for line in f if line.strip())
raw_count=count_lines(DATA)
print(f'Non-empty source rows: {raw_count:,}')
if raw_count < 1_000_000: raise ValueError(f'Need at least 1,000,000 source rows; found {raw_count:,}. No training started.')

## 2. Validate and clean JSONL
Invalid rows are skipped by the repository validator. The notebook refuses to proceed if fewer than one million valid examples remain. For better results, deduplicate near-duplicates and keep a separate held-out evaluation set before training.

In [ ]:
CLEAN='/content/drive/MyDrive/mj-ai-data/mj_ai_1m_clean.jsonl'
subprocess.run(['python',f'{REPO}/training/prepare_dataset.py',DATA,CLEAN],check=True)
clean_count=count_lines(CLEAN)
print(f'Valid examples: {clean_count:,}')
if clean_count < 1_000_000: raise ValueError(f'Only {clean_count:,} valid examples; fix dataset quality before training.')

## 3. Train LoRA
Default uses one epoch, sequence length 512 and LoRA rank 16 to keep memory use more manageable. This can still take a long time. If runtime disconnects, the saved adapter/checkpoints may be incomplete; restart from a fresh run or configure checkpoint resume deliberately. Do not claim training succeeded until the script prints SUCCESS.

In [ ]:
OUT='/content/drive/MyDrive/mj-ai-data/mj-ai-1m-lora'
if os.path.isdir(OUT):
    import shutil; shutil.rmtree(OUT)
cmd=['python',f'{REPO}/training/train_lora.py','--model','Qwen/Qwen2.5-0.5B-Instruct','--data',CLEAN,'--output',OUT,'--epochs','1','--max-length','512','--rank','16','--gradient-checkpointing']
print('Starting one-epoch training on',f'{clean_count:,}','examples')
subprocess.run(cmd,check=True)
assert os.path.isfile(os.path.join(OUT,'adapter_config.json'))
weights=[x for x in os.listdir(OUT) if x.endswith(('.safetensors','.bin'))]
assert weights, 'Adapter weights not found'
print('TRAINING VERIFIED:',OUT,weights)

## 4. Export adapter ZIP
The ZIP contains the LoRA adapter, not the full base model. Keep the base model ID the same when loading it.

In [ ]:
import zipfile
ZIP='/content/drive/MyDrive/mj-ai-data/mj-ai-1m-lora.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    for root,dirs,files in os.walk(OUT):
        for name in files:
            p=os.path.join(root,name)
            z.write(p,os.path.relpath(p,OUT))
assert os.path.getsize(ZIP)>0
print('Adapter ZIP:',ZIP,'size MB:',round(os.path.getsize(ZIP)/1024/1024,2))